# Tarea 1
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2-2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad1_Sakuragi_Mitsui_Rukagua_Sendoh.ipynb
* Subir el archivo al link de entrega Actividad 1 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 23 de agosto de 2026.

__Integrantes:__ (RUT, Nombre y Apellido)

* 13.257.556-8, Ricardo Lopez
* 16.789.149-7, Camilo Muñoz

## Librerias

In [ ]:
import os

SEED = 84
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

import random
from pathlib import Path
import zipfile
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy import linspace
from pandas import DataFrame, read_csv

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
import keras
from keras import layers, ops
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "operaciones determin?sticas habilitadas"
except Exception as exc:
    determinism_status = f"no se pudo forzar determinismo completo: {exc}"

print(f"Reproducibility settings applied with SEED={SEED}; {determinism_status}.")


## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
    """Grafica las m?tricas disponibles en un objeto History de Keras."""
    if history is None or not getattr(history, "history", None):
        print("El objeto history est? vac?o; no hay m?tricas para graficar.")
        return

    metric_names = [name for name in history.history if not name.startswith("val_")]
    if not metric_names:
        print("No se encontraron m?tricas de entrenamiento para graficar.")
        return

    epochs = range(1, len(next(iter(history.history.values()))) + 1)
    fig, axes = plt.subplots(len(metric_names), 1, figsize=(width, max(height, 2.4 * len(metric_names))))
    if len(metric_names) == 1:
        axes = [axes]

    for ax, metric_name in zip(axes, metric_names):
        ax.plot(epochs, history.history[metric_name], "o-", label=metric_name)
        val_metric_name = f"val_{metric_name}"
        if val_metric_name in history.history:
            ax.plot(epochs, history.history[val_metric_name], "o-", label=val_metric_name)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric_name)
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()


## Dataset

<center>
    <img src=https://www.xenonstack.com/hs-fs/hubfs/xenonstack-credit-card-fraud-detection.png?width=1920&height=1080&name=xenonstack-credit-card-fraud-detection.png width=800>
</center>

El conjunto de datos contiene transacciones realizadas con tarjetas de crédito en septiembre de 2013 en usuarios europeos.
Este conjunto de datos presenta transacciones que ocurrieron en dos días, donde tiene diagnosticado 492 casos fraudes de 284.807 transacciones. El conjunto de datos está muy desequilibrado, la clase positiva (fraudes) representa el 0,172% de todas las transacciones.

* La data y detalles está completamente disponible en [Kaggle: Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud)

<p style="color:red"> <b>Instrucciones para usuarios que usan Windows: </b></p>

* Descargar la data pulsando [¡¡AQUI!!](https://github.com/adoc-box/Datasets/raw/main/creditcard.zip)
* Descomprime el archivo __creditcard.zip__. Luego, dejar el archivo __'creditcard.csv__ junto con el script.
* Ya con eso, debería poder ejecutar el script sin ningun problema.

In [ ]:
DATASET_URL = "https://github.com/adoc-box/Datasets/raw/main/creditcard.zip"
DATASET_NAME = "creditcard.csv"
ZIP_NAME = "creditcard.zip"

candidate_paths = [
    Path("/content/creditcard/creditcard.csv"),
    Path("/content/creditcard.csv"),
    Path("creditcard/creditcard.csv"),
    Path("creditcard.csv"),
]


def find_dataset_path():
    for candidate in candidate_paths:
        if candidate.exists():
            return candidate
    return None


data_path = find_dataset_path()

if data_path is None:
    zip_path = Path(ZIP_NAME)
    if not zip_path.exists():
        print(f"Descargando dataset desde {DATASET_URL} ...")
        urllib.request.urlretrieve(DATASET_URL, zip_path)

    print(f"Descomprimiendo {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(".")

    data_path = find_dataset_path()

if data_path is None:
    expected = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(f"No se encontr? {DATASET_NAME}. Rutas revisadas:\n{expected}")

print(f"Dataset disponible en: {data_path}")


In [ ]:
data = read_csv(data_path)

expected_columns = {"Time", "Amount", "Class", *[f"V{i}" for i in range(1, 29)]}
missing_columns = sorted(expected_columns.difference(data.columns))
assert not missing_columns, f"Faltan columnas esperadas: {missing_columns}"
assert data.shape[0] > 0, "El dataset est? vac?o."
assert set(data["Class"].unique()) == {0, 1}, "Class debe contener exactamente las etiquetas 0 y 1."
assert not data.isna().any().any(), "El dataset contiene valores NaN."

print("No. of unique labels", data["Class"].nunique())
print("Label values", sorted(data["Class"].unique()))
print("-------")
print("Break down of the Normal and Fraud Transactions")
print(data["Class"].value_counts(sort=True), end=2 * "\n")

data.head()


In [ ]:
# La carga y validaci?n del dataset quedaron consolidadas en la celda anterior.
# Esta celda se mantiene como una vista r?pida para evitar recargar datos duplicadamente.
data.head()


In [ ]:
## Data splitting: the normal and fradulent transactions in separate dataframe
normal_data = data.query("Class == 0")
fraud_data = data.query("Class == 1")

## Visualize transaction amounts for normal and fraudulent transactions
bins = linspace(200, 2500, 100)
plt.figure(figsize=(14, 4))
plt.hist(normal_data.Amount, bins=bins, alpha=1, density=True, label='Normal')
plt.hist(fraud_data.Amount, bins=bins, alpha=0.5, density=True, label='Fraud')
plt.legend(loc='upper right')
plt.title("Transaction amount vs Percentage of transactions")
plt.xlabel("Transaction amount (USD)"); plt.ylabel("Percentage of transactions");
plt.tight_layout()
plt.show()

In [ ]:
data.describe(include='all')

### Data pre-processing

In [ ]:
X = data.drop(columns=["Time", "Class"])
Y = data["Class"].astype(int)

assert X.shape[1] == 29, f"Se esperaban 29 variables predictoras; se obtuvieron {X.shape[1]}."
assert Y.isin([0, 1]).all(), "Las etiquetas deben ser binarias: 0 normal, 1 fraude."

X_trainVal, X_test, y_trainVal, y_test = train_test_split(
    X,
    Y,
    stratify=Y,
    test_size=0.2,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainVal,
    y_trainVal,
    stratify=y_trainVal,
    test_size=0.2,
    random_state=SEED,
)

print(f"Train (Shape) - X: {X_train.shape}, y: {y_train.shape}")
print(f"Val (Shape) - X: {X_val.shape}, y: {y_val.shape}")
print(f"Test (Shape) - X: {X_test.shape}, y: {y_test.shape}\n")

X_train_normal_raw = X_train[y_train.values == 0]
X_val_normal_raw = X_val[y_val.values == 0]
X_trainVal_normal_raw = X_trainVal[y_trainVal.values == 0]

print(f"Train (Shape) Normal cases - X: {X_train_normal_raw.shape}")
print(f"Val (Shape) Normal cases - X: {X_val_normal_raw.shape}")
print(f"Train+Val (Shape) Normal cases - X: {X_trainVal_normal_raw.shape}")

scaler = MinMaxScaler().fit(X_train_normal_raw)
X_train_normal = scaler.transform(X_train_normal_raw).astype("float32")
X_val_normal = scaler.transform(X_val_normal_raw).astype("float32")
X_trainVal_normal = scaler.transform(X_trainVal_normal_raw).astype("float32")
X_train_scaled = scaler.transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

for name, array in {
    "X_train_normal": X_train_normal,
    "X_val_normal": X_val_normal,
    "X_val": X_val,
    "X_test": X_test,
}.items():
    assert np.isfinite(array).all(), f"{name} contiene valores no finitos."

assert X_train_normal.shape[0] == (y_train == 0).sum(), "El entrenamiento normal no coincide con y_train."
assert X_val.shape[0] == y_val.shape[0], "X_val e y_val no tienen la misma cantidad de filas."
assert X_test.shape[0] == y_test.shape[0], "X_test e y_test no tienen la misma cantidad de filas."


## Paso 1 (5 puntos):

Utilice el conjunto de entrenamiento y validación de la mejor forma tal de poder definir los hiperparámetros (número de capas ocultas, número de unidades en las capas, funciones de activación, número de épocas) de un modelo **Autoencoder Variacional (VAE)**  para abordar el problema de detección de fraudes siguiendo un enfoque similar al ejemplo de Vanilla Autoencoder visto en clases.

Deje las 3 mejores cofiguraciones probados reportando también las métricas MAE y MSE de los conjuntos de entrenamiento y validación. También indiquen con cuál de las 3 configuraciones eligirian como su mejor opción.

## Paso 1: Implementación y optimización del Autoencoder Variacional (VAE)

En esta sección, implementaremos un Autoencoder Variacional (VAE) utilizando TensorFlow y Keras. El VAE constará de un codificador (Encoder) que mapea los datos de entrada a un espacio latente, una capa de muestreo (Sampling Layer) que utiliza la reparametrización para generar muestras del espacio latente, y un decodificador (Decoder) que reconstruye los datos a partir de estas muestras. La pérdida total del VAE incluirá tanto la pérdida de reconstrucción como la divergencia KL para regularizar el espacio latente.

Se entrenarán al menos tres configuraciones diferentes del VAE, variando hiperparámetros como el número de capas ocultas, unidades, dimensión latente, función de activación, tamaño del batch y número de épocas. Utilizaremos EarlyStopping para detener el entrenamiento cuando la pérdida de validación no mejore y restaurar los pesos del mejor modelo. Los resultados de cada configuración serán tabulados para identificar la mejor opción.

In [ ]:
class Sampling(layers.Layer):
    """Capa de reparametrizaci?n para el espacio latente del VAE."""

    def __init__(self, seed=SEED, **kwargs):
        super().__init__(**kwargs)
        self.seed_generator = keras.random.SeedGenerator(seed)

    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = keras.random.normal(shape=ops.shape(z_mean), seed=self.seed_generator)
        return z_mean + ops.exp(0.5 * z_log_var) * epsilon


def build_encoder(input_dim, hidden_layers, latent_dim, activation):
    encoder_inputs = keras.Input(shape=(input_dim,), name="encoder_input")
    x = encoder_inputs
    for units in hidden_layers:
        x = layers.Dense(units, activation=activation)(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling(name="sampling")([z_mean, z_log_var])
    return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")


def build_decoder(input_dim, hidden_layers, latent_dim, activation):
    latent_inputs = keras.Input(shape=(latent_dim,), name="z_sampling")
    x = latent_inputs
    for units in reversed(hidden_layers):
        x = layers.Dense(units, activation=activation)(x)
    decoder_outputs = layers.Dense(input_dim, activation="sigmoid", name="decoder_output")(x)
    return keras.Model(latent_inputs, decoder_outputs, name="decoder")


class VAE(keras.Model):
    """VAE compatible con Keras 3 mediante train_step/test_step personalizados."""

    def __init__(self, encoder, decoder, kl_weight=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.kl_weight = kl_weight
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs, training=False):
        z_mean, _, z = self.encoder(inputs, training=training)
        latent = z if training else z_mean
        return self.decoder(latent, training=training)

    def compute_losses(self, x, training):
        z_mean, z_log_var, z = self.encoder(x, training=training)
        latent = z if training else z_mean
        reconstruction = self.decoder(latent, training=training)
        reconstruction_loss = ops.mean(ops.sum(ops.square(x - reconstruction), axis=1))
        kl_loss = -0.5 * ops.mean(
            ops.sum(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var), axis=1)
        )
        total_loss = reconstruction_loss + self.kl_weight * kl_loss
        return total_loss, reconstruction_loss, kl_loss

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            total_loss, reconstruction_loss, kl_loss = self.compute_losses(x, training=True)
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        total_loss, reconstruction_loss, kl_loss = self.compute_losses(x, training=False)
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {metric.name: metric.result() for metric in self.metrics}


def create_vae_model(input_dim, hidden_layers, latent_dim, activation, learning_rate, kl_weight):
    encoder = build_encoder(input_dim, hidden_layers, latent_dim, activation)
    decoder = build_decoder(input_dim, hidden_layers, latent_dim, activation)
    vae = VAE(encoder, decoder, kl_weight=kl_weight, name="vae_mlp")
    vae.compile(optimizer=Adam(learning_rate=learning_rate))
    return vae, encoder, decoder


def reconstruction_metrics(model, array):
    reconstructed = model.predict(array, verbose=0)
    mae = np.mean(np.abs(array - reconstructed))
    mse = np.mean(np.square(array - reconstructed))
    return mae, mse


configurations = [
    {
        "name": "Config_1 (Small)",
        "hidden_layers": [16, 8],
        "latent_dim": 4,
        "activation": "relu",
        "epochs": 50,
        "batch_size": 256,
        "learning_rate": 1e-3,
        "kl_weight": 1e-3,
    },
    {
        "name": "Config_2 (Medium)",
        "hidden_layers": [32, 16],
        "latent_dim": 8,
        "activation": "relu",
        "epochs": 50,
        "batch_size": 512,
        "learning_rate": 1e-3,
        "kl_weight": 1e-3,
    },
    {
        "name": "Config_3 (Large)",
        "hidden_layers": [64, 32, 16],
        "latent_dim": 16,
        "activation": "tanh",
        "epochs": 50,
        "batch_size": 1024,
        "learning_rate": 5e-4,
        "kl_weight": 1e-3,
    },
]

results = []
input_dim = X_train_normal.shape[1]

for config_id, config in enumerate(configurations):
    print(f"\nTraining {config['name']}...")
    vae, encoder, decoder = create_vae_model(
        input_dim=input_dim,
        hidden_layers=config["hidden_layers"],
        latent_dim=config["latent_dim"],
        activation=config["activation"],
        learning_rate=config["learning_rate"],
        kl_weight=config["kl_weight"],
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
        mode="min",
    )

    history = vae.fit(
        X_train_normal,
        X_train_normal,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=True,
        validation_data=(X_val_normal, X_val_normal),
        callbacks=[early_stopping],
        verbose=0,
    )

    train_mae, train_mse = reconstruction_metrics(vae, X_train_normal)
    val_mae, val_mse = reconstruction_metrics(vae, X_val_normal)

    result = {
        "config_id": config_id,
        "name": config["name"],
        "hidden_layers": config["hidden_layers"],
        "latent_dim": config["latent_dim"],
        "activation": config["activation"],
        "epochs": len(history.history["loss"]),
        "batch_size": config["batch_size"],
        "learning_rate": config["learning_rate"],
        "kl_weight": config["kl_weight"],
        "train_mae": train_mae,
        "train_mse": train_mse,
        "val_mae": val_mae,
        "val_mse": val_mse,
        "model": vae,
        "history": history,
    }
    results.append(result)
    print(f"{config['name']} - Train MAE: {train_mae:.4f}, Val MAE: {val_mae:.4f}")

results_df = DataFrame(results).drop(columns=["model", "history"])
results_df_sorted = results_df.sort_values(by="val_mae").reset_index(drop=True)

print("\n--- VAE Configurations Results ---")
display(results_df_sorted)

best_result = min(results, key=lambda item: item["val_mae"])
best_vae = best_result["model"]
best_vae_history = best_result["history"]
best_config = {key: best_result[key] for key in ["name", "hidden_layers", "latent_dim", "activation", "batch_size", "learning_rate", "kl_weight"]}

print("\nCriterio de selecci?n: menor MAE de reconstrucci?n en validaci?n normal.")
print(f"Configuraci?n ganadora: {best_result['name']}")

plot_history(best_vae_history)

print("\nNota sobre reproducibilidad: se fijaron semillas y operaciones determin?sticas. Aun as?, pueden existir peque?as variaciones por hardware/GPU y operaciones de punto flotante.")


## Paso 2: Cálculo del error de reconstrucción y optimización del umbral

En esta sección, utilizaremos el VAE ganador del Paso 1 para calcular los errores de reconstrucción sobre el conjunto de validación completo (incluyendo transacciones normales y fraudulentas). El objetivo es encontrar un umbral óptimo para el error de reconstrucción que maximice el F1-score, una métrica adecuada para datasets desbalanceados como este. Se utilizará la curva de precisión-recall para identificar este umbral, y se presentarán las métricas resultantes junto con una visualización de la distribución de errores para ambos tipos de transacciones.

In [ ]:
y_val_true = y_val.to_numpy()

X_val_pred = best_vae.predict(X_val, verbose=0)
reconstruction_error_val = np.mean(np.abs(X_val - X_val_pred), axis=1)

precision, recall, thresholds = precision_recall_curve(y_val_true, reconstruction_error_val)
precision_for_thresholds = precision[:-1]
recall_for_thresholds = recall[:-1]

f1_scores = np.divide(
    2 * precision_for_thresholds * recall_for_thresholds,
    precision_for_thresholds + recall_for_thresholds,
    out=np.zeros_like(thresholds, dtype=float),
    where=(precision_for_thresholds + recall_for_thresholds) > 0,
)

best_f1_idx = int(np.nanargmax(f1_scores))
best_threshold = float(thresholds[best_f1_idx])

# Una transacci?n se marca como fraude cuando su error de reconstrucci?n supera el umbral.
y_pred_val = (reconstruction_error_val >= best_threshold).astype(int)
f1_val = f1_score(y_val_true, y_pred_val, zero_division=0)
precision_val = precision_score(y_val_true, y_pred_val, zero_division=0)
recall_val = recall_score(y_val_true, y_pred_val, zero_division=0)

print("\n--- Resultados de optimizaci?n de umbral en validaci?n ---")
print(f"Mejor umbral (maximizaci?n F1-score): {best_threshold:.6f}")
print(f"F1-score en validaci?n: {f1_val:.4f}")
print(f"Precisi?n en validaci?n: {precision_val:.4f}")
print(f"Recall en validaci?n: {recall_val:.4f}")

error_normal_val = reconstruction_error_val[y_val_true == 0]
error_fraud_val = reconstruction_error_val[y_val_true == 1]

plt.figure(figsize=(10, 6))
plt.hist(error_normal_val, bins=50, density=True, alpha=0.6, label="Normal (Validaci?n)")
plt.hist(error_fraud_val, bins=50, density=True, alpha=0.6, label="Fraude (Validaci?n)")
plt.axvline(best_threshold, color="red", linestyle="dashed", linewidth=2, label=f"Umbral ?ptimo: {best_threshold:.6f}")
plt.title("Distribuci?n del error de reconstrucci?n en validaci?n")
plt.xlabel("Error de reconstrucci?n (MAE)")
plt.ylabel("Densidad")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Paso 3: Evaluación del modelo en el conjunto de prueba y conclusiones

Finalmente, evaluaremos el rendimiento del modelo VAE ganador junto con el umbral óptimo encontrado en el Paso 2, utilizando el conjunto de prueba completamente independiente (`X_test`). Se generará la matriz de confusión y un informe de clasificación detallado para analizar las métricas de Precision, Recall y F1-score para la clase fraudulenta. Se incluirá una breve conclusión para interpretar los resultados, destacando la importancia de métricas distintas a la precisión en este contexto y las limitaciones del enfoque.

In [ ]:
y_test_true = y_test.to_numpy()

X_test_pred = best_vae.predict(X_test, verbose=0)
reconstruction_error_test = np.mean(np.abs(X_test - X_test_pred), axis=1)
y_pred_test = (reconstruction_error_test >= best_threshold).astype(int)

print("\n--- Evaluaci?n del modelo en el conjunto de prueba ---")
print(f"Umbral utilizado: {best_threshold:.6f}")

cm = confusion_matrix(y_test_true, y_pred_test)
tn, fp, fn, tp = cm.ravel()

display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Fraude"])
fig, ax = plt.subplots(figsize=(6, 6))
display_cm.plot(ax=ax, cmap=plt.cm.Blues, values_format="d")
plt.title("Matriz de confusi?n en el conjunto de prueba")
plt.show()

report = classification_report(
    y_test_true,
    y_pred_test,
    target_names=["Normal", "Fraude"],
    zero_division=0,
)
report_dict = classification_report(
    y_test_true,
    y_pred_test,
    target_names=["Normal", "Fraude"],
    zero_division=0,
    output_dict=True,
)

precision_fraud = precision_score(y_test_true, y_pred_test, pos_label=1, zero_division=0)
recall_fraud = recall_score(y_test_true, y_pred_test, pos_label=1, zero_division=0)
f1_fraud = f1_score(y_test_true, y_pred_test, pos_label=1, zero_division=0)

print("\nMatriz de confusi?n:")
print(cm)
print("\nInforme de clasificaci?n:\n")
print(report)
print("M?tricas para la clase 'Fraude' en test:")
print(f"  Precisi?n (Fraude): {precision_fraud:.4f}")
print(f"  Recall (Fraude): {recall_fraud:.4f}")
print(f"  F1-score (Fraude): {f1_fraud:.4f}")
print(f"  FP: {fp}")
print(f"  FN: {fn}")


## Conclusi?n de la Detecci?n de Fraude con VAE

La siguiente celda genera una conclusi?n con los valores reales obtenidos en test. Debe ejecutarse despu?s del Paso 3, porque usa la matriz de confusi?n y las m?tricas calculadas all?.


In [ ]:
from IPython.display import Markdown, display

priority = "detectar una mayor proporci?n de fraudes" if recall_fraud >= precision_fraud else "reducir falsas alarmas"

conclusion = f"""
La detecci?n de fraude con tarjetas de cr?dito es un problema fuertemente desbalanceado: la clase fraudulenta representa una fracci?n muy peque?a de las transacciones. Por eso, la accuracy global no es una m?trica suficiente; un modelo que clasifique casi todo como normal puede verse muy preciso y aun as? fallar en el objetivo principal.

Con la configuraci?n seleccionada ({best_config['name']}) y el umbral optimizado en validaci?n ({best_threshold:.6f}), el desempe?o en test para la clase fraude fue:

- Falsos positivos (FP): {fp}
- Falsos negativos (FN): {fn}
- Precisi?n fraude: {precision_fraud:.4f}
- Recall fraude: {recall_fraud:.4f}
- F1-score fraude: {f1_fraud:.4f}

El modelo tiende a priorizar {priority}. En t?rminos pr?cticos, el recall indica qu? proporci?n de fraudes reales fueron detectados, mientras que la precisi?n indica qu? proporci?n de alertas de fraude fueron efectivamente fraude.

Como limitaci?n, el VAE aprende principalmente la estructura de las transacciones normales y decide por error de reconstrucci?n. Si algunas transacciones fraudulentas se parecen mucho a operaciones normales poco frecuentes, pueden reconstruirse bien y quedar como falsos negativos. Para mejorar el resultado, se podr?an explorar m?s configuraciones, ajustar el peso KL, comparar contra autoencoders no variacionales y evaluar umbrales seg?n el costo de FP y FN.
"""

display(Markdown(conclusion))


In [ ]:
# Celda reservada para notas o experimentos adicionales.


## Paso 2 (3 puntos):

Con el mejor modelo entrenado en el Paso 1, encuentre el mejor umbral para el error tal que pueda maximizar la métrica F1-Score en el conjunto de validación.

In [ ]:
# Celda reservada para notas o experimentos adicionales.


In [ ]:
# Celda reservada para notas o experimentos adicionales.


## Paso 3 (2 puntos):

Usando el modelo entrenado y el umbral encontrado en los pasos anteriores, evaluar el modelo con el conjunto de prueba y entregue la matriz de confusión y el __classification_report__.

In [ ]:
# Celda reservada para notas o experimentos adicionales.


In [ ]:
# Celda reservada para notas o experimentos adicionales.
